In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_data = pd.read_csv("/kaggle/input/playground-series-s5e9/train.csv", index_col=0)

In [ ]:
train_data.head(20)

In [ ]:
train_data['BeatsPerMinute'].value_counts()

In [ ]:
train_data.describe()

In [ ]:
train_data.info()

In [ ]:
train_data['BeatsPerMinute'].value_counts(normalize=True)

In [ ]:
# Target Variable Analysis

plt.figure(figsize=(8, 6))

sns.histplot(train_data['BeatsPerMinute'], kde=True, bins=30)
plt.title("Distribution of BeatsPerMinute")
plt.show()

In [ ]:
train_data.isnull().sum()

In [ ]:
# Feature Distribution

numeric_features = train_data.select_dtypes(include=[np.number]).columns.tolist()

train_data[numeric_features].hist(bins=30, figsize=(15,12), layout=(4,3))
plt.suptitle("Feature Distributions")
plt.show()

In [ ]:
correlation_matrix = train_data.corr()

In [ ]:
correlation_matrix

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True)

In [ ]:
print("\nCorrelation of each feature with Target (BeatsPerMinutes) feature")
correlation_matrix["BeatsPerMinute"].sort_values(ascending=False)

In [ ]:
train_data = train_data.reset_index().drop('id', axis=1)

In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

X = train_data.drop(columns = ['BeatsPerMinute'])
y = train_data["BeatsPerMinute"]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.20, random_state=42)


In [ ]:
from sklearn.linear_model import LinearRegression

linear_reg = LinearRegression()
linear_reg.fit(X_train, y_train)

In [ ]:
train_pred = linear_reg.predict(X_train)

In [ ]:
valid_pred = linear_reg.predict(X_valid)

In [ ]:
# Training Evaluation

rmse = np.sqrt(mean_squared_error(y_train, train_pred))
mae = mean_absolute_error(y_train, train_pred)
r2 = r2_score(y_train, train_pred)

print(f"Training Evalutation\n\nRMSE Score : {rmse},\nMAE Score: {mae},\nR-Squared: {r2} ")

In [ ]:
# Validation Evaluation 

rmse_valid = np.sqrt(mean_squared_error(y_valid, valid_pred))
mae = mean_absolute_error(y_valid, valid_pred)
r2 = r2_score(y_valid, valid_pred)

print(f"Validation Evalutation\n\nRMSE Score : {rmse},\nMAE Score: {mae},\nR-Squared: {r2} ")

It is evident here that the linear model is unable to capture any useful information from the given dataset. The correlation analysis also suggested that features don't have any linear relationship among them. Hence, as a next step trying to fit a non-linear model that can capture the relationship better.

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold

# xgb_model = XGBRegressor(random_state=42)
xgb_model = XGBRegressor(
    objective="reg:squarederror",  # RMSE-friendly objective
    random_state=42,
    n_jobs=2,
    eval_metric="rmse",
    early_stopping_rounds=50 
)

param_dist = {
    "n_estimators": [300, 500, 800, 1200],
    "learning_rate": [0.01, 0.02, 0.03],
    "max_depth": [5, 6, 7, 8],
    "min_child_weight": [1, 3, 5, 7],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.2, 0.5],
    "reg_alpha": [0.05, 0.1, 0.2],
    "reg_lambda": [5, 10, 15]
}

# param_list = {
#     'n_estimators':[200, 500],      # number of boosting rounds (trees)
#     'learning_rate':[0.01, 0.05, 0.1],    # step size shrinkage
#     'subsample':[0.6, 1.0],         # fraction of samples used per tree
#     'colsample_bytree':[0.7, 1.0],
# }

# random_search = RandomizedSearchCV(
# estimator=xgb_model,
# param_distributions=param_list,
# n_iter=50,
# scoring='neg_mean_squared_error',
# cv=3,
# verbose=1,
# n_jobs=2,
# random_state=42
# )

cv = KFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=50,  # more iterations for better exploration
    scoring="neg_root_mean_squared_error",  # directly optimize RMSE
    cv=cv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

# random_search.fit(X_train, y_train)
random_search.fit(X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False)

In [ ]:
print("Best parameters:", random_search.best_params_)

In [ ]:
best_model = random_search.best_estimator_

In [ ]:
best_model_valid_pred = best_model.predict(X_valid)

In [ ]:
rmse_valid = np.sqrt(mean_squared_error(y_valid, best_model_valid_pred))
mae = mean_absolute_error(y_valid, best_model_valid_pred)
r2 = r2_score(y_valid, best_model_valid_pred)

print(f"Validation Evalutation for XGBoost\n\nRMSE Score : {rmse},\nMAE Score: {mae},\nR-Squared: {r2} ")

In [ ]:
plt.figure(figsize=(10,6))
plt.barh(X.columns, best_model.feature_importances_)
plt.title("XGBoost Feature Importance")
plt.show()

In [ ]:
test_data = pd.read_csv("/kaggle/input/playground-series-s5e9/test.csv", index_col=0)

In [ ]:
test_data.head()

In [ ]:
X_test = test_data.reset_index().drop('id', axis=1)

In [ ]:
y_pred_test = best_model.predict(X_test)

In [ ]:
output = pd.DataFrame({
    "id": test_data.index,
    "Predicted_BPM": y_pred_test
})

output.to_csv("submission.csv", index=False)


In [ ]:
output.head()